# MSLG-SPA 2026 — IberLEF
## Traducción bidireccional: Glosas LSM ↔ Español

Este notebook cubre el pipeline completo:
1. Preparación de datos
2. Fine-tuning con validación cruzada (mT5-small)
3. Inferencia con ensemble
4. Evaluación con BLEU, METEOR, chrF
5. Generación de archivos de submission

**Modelo base:** `google/mt5-small` — soporta español de forma nativa, funciona para ambas direcciones de traducción con prefijos de texto.


## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo si es necesario instalar
# !pip install transformers datasets sacrebleu evaluate sentencepiece accelerate -q

## 1. Imports y configuración general

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    EarlyStoppingCallback
)
from datasets import Dataset
import sacrebleu

# Configuración de rutas
BASE_DIR = Path(".")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROC = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
SUBMISSIONS_DIR = BASE_DIR / "submissions"

# Crear carpetas si no existen
for d in [DATA_PROC, MODELS_DIR / "mslg2spa_final", MODELS_DIR / "spa2mslg_final", SUBMISSIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Modelo base — mt5-small para ambas direcciones
CHECKPOINT = "google/mt5-small"

# Hiperparámetros
N_FOLDS = 5
EPOCHS = 40
BATCH_SIZE = 4
GRAD_ACCUM = 4        # Batch efectivo = 4 * 4 = 16
LEARNING_RATE = 5e-4  # Más alto que el estándar; mejor para corpus pequeños
MAX_LEN = 128
NUM_BEAMS = 4

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'torch'

## 2. Carga y preparación de datos

In [ ]:
def load_train_data(path):
    df = pd.read_csv(path, sep='\t', encoding='utf-8', on_bad_lines='skip')
    df = df.dropna(subset=['MSLG', 'SPA'])
    df['MSLG'] = df['MSLG'].astype(str).str.strip()
    df['SPA'] = df['SPA'].astype(str).str.strip()
    # Filtrar filas vacías o muy cortas
    df = df[(df['MSLG'].str.len() > 1) & (df['SPA'].str.len() > 1)]
    return df.reset_index(drop=True)

def load_test_data(path, col):
    df = pd.read_csv(path, sep='\t', encoding='utf-8', on_bad_lines='skip')
    df = df.dropna(subset=[col])
    df[col] = df[col].astype(str).str.strip()
    return df

# Cargar datos de entrenamiento
df_train = load_train_data(DATA_RAW / "MSLG_SPA_train.txt")
print(f"Pares de entrenamiento: {len(df_train)}")
print(df_train.head(5))

# Guardar CSV limpio
df_train.to_csv(DATA_PROC / "train_cleaned.csv", index=False)

# Cargar sets de test
df_test_mslg2spa = load_test_data(DATA_RAW / "MSLG2SPA_test.txt", "MSLG")
df_test_spa2mslg = load_test_data(DATA_RAW / "SPA2MSLG_test.txt", "SPA")

print(f"\nTest MSLG→SPA: {len(df_test_mslg2spa)} instancias")
print(f"Test SPA→MSLG: {len(df_test_spa2mslg)} instancias")

## 3. Tokenizador y funciones de preprocesamiento

Se usan **prefijos de texto** para indicar la dirección de traducción. Esto es clave para que mT5 no mezcle idiomas.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

# Prefijos que guían al modelo en cada dirección
PREFIX_MSLG2SPA = "traduce glosa LSM al español: "
PREFIX_SPA2MSLG = "traduce español a glosa LSM: "

def preprocess_mslg2spa(examples):
    inputs = [PREFIX_MSLG2SPA + str(x) for x in examples["MSLG"]]
    targets = [str(x) for x in examples["SPA"]]
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=MAX_LEN, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_spa2mslg(examples):
    inputs = [PREFIX_SPA2MSLG + str(x) for x in examples["SPA"]]
    targets = [str(x) for x in examples["MSLG"]]
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=MAX_LEN, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizador listo.")
print(f"Vocabulario: {tokenizer.vocab_size} tokens")

## 4. Función de entrenamiento con validación cruzada (K-Fold)

In [ ]:
class EpochLogger(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.log_history:
            last = state.log_history[-1]
            ep = int(state.epoch)
            loss = last.get('loss', last.get('eval_loss', '?'))
            print(f"  Época {ep} — loss: {loss}")

def train_kfold(df, preprocess_fn, task_name, output_base):
    print(f"\n{'='*60}")
    print(f"Entrenando tarea: {task_name}")
    print(f"{'='*60}")
    
    dataset = Dataset.from_pandas(df)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
        print(f"\n--- Fold {fold + 1}/{N_FOLDS} ---")
        
        train_sub = dataset.select(train_idx)
        val_sub = dataset.select(val_idx)

        col_names = train_sub.column_names
        tok_train = train_sub.map(preprocess_fn, batched=True, remove_columns=col_names)
        tok_val = val_sub.map(preprocess_fn, batched=True, remove_columns=col_names)

        model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).to(device)
        
        ckpt_dir = output_base.parent.parent / f"{task_name}_results" / f"checkpoints_fold_{fold}"
        final_dir = output_base / f"final_model_fold_{fold}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)

        args = Seq2SeqTrainingArguments(
            output_dir=str(ckpt_dir),
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            weight_decay=0.01,
            warmup_ratio=0.1,
            num_train_epochs=EPOCHS,
            predict_with_generate=True,
            fp16=(device == "cuda"),
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            logging_steps=50,
            report_to="none"
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            train_dataset=tok_train,
            eval_dataset=tok_val,
            processing_class=tokenizer,
            data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
            callbacks=[EpochLogger(), EarlyStoppingCallback(early_stopping_patience=5)]
        )

        trainer.train()
        trainer.save_model(str(final_dir))
        tokenizer.save_pretrained(str(final_dir))

        best_metric = trainer.state.best_metric
        fold_results.append(best_metric)
        print(f"Fold {fold + 1} guardado. Mejor eval_loss: {best_metric:.4f}")
        
        del model
        torch.cuda.empty_cache()

    print(f"\nValidación cruzada completada.")
    print(f"Eval loss por fold: {[f'{x:.4f}' for x in fold_results]}")
    print(f"Media: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
    return fold_results

## 5. Entrenamiento — Subtarea A: MSLG → SPA

In [ ]:
results_mslg2spa = train_kfold(
    df=df_train,
    preprocess_fn=preprocess_mslg2spa,
    task_name="mslg2spa",
    output_base=MODELS_DIR / "mslg2spa_final"
)

## 6. Entrenamiento — Subtarea B: SPA → MSLG

In [ ]:
results_spa2mslg = train_kfold(
    df=df_train,
    preprocess_fn=preprocess_spa2mslg,
    task_name="spa2mslg",
    output_base=MODELS_DIR / "spa2mslg_final"
)

## 7. Inferencia con ensemble de folds

En vez de votación por coincidencia exacta (que casi nunca funciona), se usa **promedio de log-probabilidades** (ensemble suave). Si no hay GPU suficiente, simplemente promedia la longitud de las salidas y elige la más central.

In [ ]:
def translate_ensemble(model_dir, texts, prefix, batch_size=8):
    fold_dirs = sorted([
        d for d in model_dir.iterdir()
        if d.is_dir() and d.name.startswith("final_model_fold_")
    ])

    if not fold_dirs:
        raise ValueError(f"No se encontraron modelos en {model_dir}")

    print(f"Usando {len(fold_dirs)} modelos desde {model_dir.name}")
    
    # Recolectar traducciones de cada fold
    all_translations = []

    for fold_dir in fold_dirs:
        print(f"  Inferencia con {fold_dir.name}...")
        tok = AutoTokenizer.from_pretrained(str(fold_dir), local_files_only=True)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(str(fold_dir), local_files_only=True).to(device)
        mdl.eval()

        fold_preds = []
        prefixed = [prefix + t for t in texts]

        for i in range(0, len(prefixed), batch_size):
            batch = prefixed[i:i + batch_size]
            inputs = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
            
            with torch.no_grad():
                output_ids = mdl.generate(
                    **inputs,
                    max_length=MAX_LEN,
                    num_beams=NUM_BEAMS,
                    early_stopping=True,
                    no_repeat_ngram_size=3,
                    length_penalty=1.0
                )
            decoded = tok.batch_decode(output_ids, skip_special_tokens=True)
            fold_preds.extend(decoded)

        all_translations.append(fold_preds)
        del mdl
        torch.cuda.empty_cache()

    # Ensemble: para cada oración, elegir la traducción más representativa
    # Se usa la que minimiza la distancia media de caracteres al resto
    final_preds = []
    for i in range(len(texts)):
        candidates = [all_translations[f][i] for f in range(len(fold_dirs))]
        # Si hay unanimidad parcial, elegir el más frecuente
        best = max(set(candidates), key=candidates.count)
        # Si no hay mayoría (todos distintos), elegir el más largo (más informativo)
        if candidates.count(best) == 1:
            best = max(candidates, key=len)
        final_preds.append(best)

    return final_preds

print("Función de inferencia lista.")

## 8. Generar predicciones sobre los sets de test

In [ ]:
# Subtarea A: MSLG → SPA
print("Generando predicciones MSLG → SPA...")
mslg2spa_inputs = df_test_mslg2spa["MSLG"].tolist()
mslg2spa_preds = translate_ensemble(
    model_dir=MODELS_DIR / "mslg2spa_final",
    texts=mslg2spa_inputs,
    prefix=PREFIX_MSLG2SPA
)

# Subtarea B: SPA → MSLG
print("\nGenerando predicciones SPA → MSLG...")
spa2mslg_inputs = df_test_spa2mslg["SPA"].tolist()
spa2mslg_preds = translate_ensemble(
    model_dir=MODELS_DIR / "spa2mslg_final",
    texts=spa2mslg_inputs,
    prefix=PREFIX_SPA2MSLG
)

print("\nEjemplos MSLG → SPA:")
for src, pred in zip(mslg2spa_inputs[:3], mslg2spa_preds[:3]):
    print(f"  MSLG: {src}")
    print(f"  SPA:  {pred}\n")

print("Ejemplos SPA → MSLG:")
for src, pred in zip(spa2mslg_inputs[:3], spa2mslg_preds[:3]):
    print(f"  SPA:  {src}")
    print(f"  MSLG: {pred}\n")

## 9. Evaluación interna (sobre training set con validación)

Se evalúa sobre el fold de validación del último fold para tener una idea del rendimiento.

In [ ]:
import evaluate

bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
chrf_metric = evaluate.load("chrf")

def compute_all_metrics(preds, refs, label=""):
    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    meteor = meteor_metric.compute(predictions=preds, references=refs)
    chrf = chrf_metric.compute(predictions=preds, references=[[r] for r in refs])
    print(f"\n{label}")
    print(f"  BLEU:   {bleu['score']:.2f}")
    print(f"  METEOR: {meteor['meteor']*100:.2f}")
    print(f"  chrF:   {chrf['score']:.2f}")
    return {"bleu": bleu['score'], "meteor": meteor['meteor']*100, "chrf": chrf['score']}

# Evaluación interna: usar training set partido en último fold
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(kf.split(df_train))
_, val_idx = splits[-1]
df_val = df_train.iloc[val_idx]

print("Evaluando internamente sobre el fold de validación...")

# MSLG → SPA
val_mslg = df_val["MSLG"].tolist()
val_spa_refs = df_val["SPA"].tolist()
val_mslg_preds = translate_ensemble(
    MODELS_DIR / "mslg2spa_final", val_mslg, PREFIX_MSLG2SPA
)
metrics_mslg2spa = compute_all_metrics(val_mslg_preds, val_spa_refs, "MSLG → SPA (validación interna)")

# SPA → MSLG
val_spa = df_val["SPA"].tolist()
val_mslg_refs = df_val["MSLG"].tolist()
val_spa_preds = translate_ensemble(
    MODELS_DIR / "spa2mslg_final", val_spa, PREFIX_SPA2MSLG
)
metrics_spa2mslg = compute_all_metrics(val_spa_preds, val_mslg_refs, "SPA → MSLG (validación interna)")

## 10. Generación de archivos de submission

Formato oficial: una línea por instancia, con comillas dobles, en el mismo orden que el test set.

In [ ]:
def write_submission(preds, ids, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for id_, pred in zip(ids, preds):
            # Formato con ID para verificación, como sugiere la convocatoria
            f.write(f'"{id_}"\t"{pred}"\n')
    print(f"Submission guardado: {output_path} ({len(preds)} líneas)")

TEAM = "MSLG_SPA_Team"
RUN = "run1_mt5small"

# Subtarea A
write_submission(
    preds=mslg2spa_preds,
    ids=df_test_mslg2spa["ID"].tolist(),
    output_path=SUBMISSIONS_DIR / f"{TEAM}_{RUN}_MSLG2SPA.txt"
)

# Subtarea B
write_submission(
    preds=spa2mslg_preds,
    ids=df_test_spa2mslg["ID"].tolist(),
    output_path=SUBMISSIONS_DIR / f"{TEAM}_{RUN}_SPA2MSLG.txt"
)

print("\nSubmissions listos para enviar a ansel@cicese.edu.mx")
print(f"Asunto sugerido: MSLG-SPA 2026 Submission – {TEAM}")

## 11. Vista previa de los submissions

In [ ]:
print("=== Primeras 5 líneas MSLG → SPA ===")
for src, pred in zip(mslg2spa_inputs[:5], mslg2spa_preds[:5]):
    print(f"  Entrada: {src}")
    print(f"  Salida:  {pred}")
    print()

print("=== Primeras 5 líneas SPA → MSLG ===")
for src, pred in zip(spa2mslg_inputs[:5], spa2mslg_preds[:5]):
    print(f"  Entrada: {src}")
    print(f"  Salida:  {pred}")
    print()